# PubMedBERT 5-Fold Training (Colab A100)

Trains `microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext` (the renamed PubMedBERT) on the medical abstracts dataset, 5-fold StratifiedKFold, class-weighted CE, bf16.

**Before running:**
1. Upload the project folder `Kaggle_new/` (containing `kaggle_trainset.csv`, `kaggle_testset.csv`, `kaggle_testset_submission.csv`, `src/`) to Google Drive at `MyDrive/Kaggle_new/`.
2. Set runtime to **A100 GPU**.
3. Run all cells. Each fold writes results to `MyDrive/Kaggle_new/outputs/bert_runs/<tag>/`.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
import os, sys
PROJECT = '/content/drive/MyDrive/Kaggle_new'
assert os.path.isdir(PROJECT), f'Project dir not found: {PROJECT}'
os.chdir(PROJECT)
sys.path.insert(0, os.path.join(PROJECT, 'src'))
# Cache HuggingFace downloads on Drive to survive session restarts.
os.environ['HF_HOME'] = os.path.join(PROJECT, 'hf_cache')
os.makedirs(os.environ['HF_HOME'], exist_ok=True)
print('CWD:', os.getcwd())
print(os.listdir())

In [ ]:
# Pinned versions known to work on Colab A100.
!pip install -q -U "transformers>=4.44,<4.50" "accelerate>=0.33" "datasets>=2.20" "scikit-learn>=1.4"

In [ ]:
import torch
print('CUDA:', torch.cuda.is_available())
print('Device:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'CPU')
print('bf16 supported:', torch.cuda.is_bf16_supported() if torch.cuda.is_available() else False)

## Smoke test — one fold, one epoch
Confirms the pipeline runs before launching the full sweep.

In [ ]:
!python src/train_bert.py --fold 0 --seed 42 --smoke --tag smoke_test

## Full 5-fold sweep — PubMedBERT base, seed=42
Each fold ≈ 8–12 min on A100 → total ~50 min.

In [ ]:
MODEL = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
SEED = 42
for fold in range(5):
    cmd = (
        f'python src/train_bert.py '
        f'--model {MODEL} --fold {fold} --seed {SEED} '
        f'--epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 '
        f'--tag pubmedbert_base_seed{SEED}_fold{fold}'
    )
    print('>>>', cmd)
    rc = os.system(cmd)
    assert rc == 0, f'fold {fold} failed'

## Summarise OOF Macro F1 across folds

In [ ]:
import json, glob, numpy as np, pandas as pd
runs = sorted(glob.glob('outputs/bert_runs/pubmedbert_base_seed42_fold*/metrics.json'))
rows = [json.load(open(p)) for p in runs]
df = pd.DataFrame(rows)
print(df[['fold', 'val_macro_f1', 'train_secs']])
print(f'\nMean OOF Macro F1: {df.val_macro_f1.mean():.4f} (std {df.val_macro_f1.std():.4f})')

## Optional: extra seeds for ensemble
Uncomment to run additional seeds. Each adds ~50 min.

In [ ]:
# MODEL = 'microsoft/BiomedNLP-BiomedBERT-base-uncased-abstract-fulltext'
# for SEED in (2024, 7):
#     for fold in range(5):
#         cmd = (
#             f'python src/train_bert.py '
#             f'--model {MODEL} --fold {fold} --seed {SEED} '
#             f'--epochs 4 --batch-size 32 --lr 2e-5 --max-length 512 '
#             f'--tag pubmedbert_base_seed{SEED}_fold{fold}'
#         )
#         print('>>>', cmd)
#         rc = os.system(cmd)
#         assert rc == 0, f'seed {SEED} fold {fold} failed'